## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 119 samples and 89 columns.
First few rows of the data:


,dmfs time,dmfs event,sample_title,gender,sample type,irdm subtype,pam50 subtype,immunity group,proliferation group,er status rna,...,MIA,MYBL2,UBE2C,AURKA,MMP11,SF3A1,CCDC157,APOBEC3G,IL2RG,SH2D1A
0,1.916667,0,SH_TS_BC113,female,Solid tumor,Basal,Basal,iweak,high,ER-,...,8.669655,0.0,10.785072,9.701948,0.0,0.0,0.0,0.0,7.698076,0.000000
1,6.750000,0,SH_TS_BC119,female,Solid tumor,HER2E,HER2E,istrong,high,ER-,...,5.828747,0.0,9.761326,9.029801,0.0,0.0,0.0,0.0,8.699401,6.999987
2,3.416667,0,SH_TS_BC146,female,Solid tumor,Basal,Basal,iweak,high,ER+,...,7.661877,0.0,10.722630,9.294487,0.0,0.0,0.0,0.0,7.882371,5.965590
3,9.750000,0,SH_TS_BC147,female,Solid tumor,LumB,LumB,iweak,high,ER+,...,6.023581,0.0,9.533534,8.878157,0.0,0.0,0.0,0.0,2.705913,7.212216
4,9.083333,0,SH_TS_BC150,female,Solid tumor,LumA,LumA,istrong,low,ER+,...,7.671198,0.0,8.358780,8.501992,0.0,0.0,0.0,0.0,8.094774,7.722999


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


/Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/.venv/lib/python3.14/site-packages/single_sample_gsea/ssgsea.py:51: RuntimeWarning: invalid value encountered in divide
  signature_ecdf = signature_ecdf / np.sum(signature_ecdf, axis=0)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
